In [ ]:
# Add project root to sys.path for clean imports
import sys
from pathlib import Path
project_root = str(Path("../..").resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.metaphor.metaphor_domain_extractor import load_usas_tags, build_analysis_prompt, analyze_text

# Test the import
usas_tags = load_usas_tags()
print(f"Loaded {len(usas_tags)} USAS tags")

# Test parameters
test_text = "Cancer is like a journey through unknown territory. The treatment hits the bad cells like a targeted missile."
target_tenor_tag = "B2"  # Health and disease

# Generate the full prompt
full_prompt = build_analysis_prompt(test_text, target_tenor_tag)

print("="*80)
print("FULL PROMPT WITH VALUES FILLED:")
print("="*80)
print(full_prompt)

print("\n" + "="*80)
print("PROMPT ANALYSIS:")
print("="*80)
print(f"Text to analyze: {test_text}")
print(f"Target tenor tag: {target_tenor_tag} ({usas_tags.get(target_tenor_tag, 'Unknown')})")
print(f"Prompt length: {len(full_prompt)} characters")
print(f"Estimated tokens: ~{len(full_prompt)//4}")

# Show first few USAS tags being used
print(f"\nFirst 10 USAS tags in the prompt:")
sample_tags = dict(list(usas_tags.items())[:10])
for code, desc in sample_tags.items():
    print(f"  {code}: {desc}")
print(f"  ... and {len(usas_tags)-10} more tags")


In [ ]:
# Cell 1: Import libraries and load data
import pandas as pd
from tqdm.auto import tqdm
from scripts.metaphor.metaphor_domain_extractor import load_usas_tags, analyze_text
import aisuite as ai
import json

# Initialize tqdm for pandas
tqdm.pandas()

# Load the JSON file into a pandas DataFrame
df = pd.read_json("/work/speech/Metaphor data/data/kanker.nl/data-export-blogs.json")

# Print the first 5 entries to see the structure
print("Dataset structure:")
print(df.head())
print(f"\nTotal entries: {len(df)}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Cell 2: Setup for metaphor analysis
# Load USAS tags and initialize client
usas_tags = load_usas_tags()
client = ai.Client()
client.configure({"ollama": {"timeout": 600}})

print(f"Loaded {len(usas_tags)} USAS tags")
print("AI client initialized")

# Set parameters
num_rows_to_process = 100
target_usas_tag = "B2"  # Health and disease tag
model = "ollama:gemma3:27b"  # or "openai:gpt-4"

print(f"Processing {num_rows_to_process} entries")
print(f"Target USAS tag: {target_usas_tag} ({usas_tags.get(target_usas_tag, 'Unknown')})")
print(f"Using model: {model}")

In [ ]:
# Cell 3: Run metaphor analysis directly on post_body
df_slice = df.head(num_rows_to_process).copy()

# Apply metaphor analysis directly to post_body with progress bar
def analyze_post_for_metaphors(text):
    """Wrapper function to call the analysis tool and handle potential errors."""
    try:
        # Pass the correct arguments to analyze_text
        return analyze_text(client, model, text, target_usas_tag)
    except Exception as e:
        # Return a dictionary with error information for debugging
        return {
            'error': str(e),
            'metaphor_present': False,
            'text_preview': text[:100] + "..." if len(text) > 100 else text
        }

print("Analyzing posts for metaphors...")
# Use .loc to avoid SettingWithCopyWarning
df_slice.loc[:, 'metaphor_analysis'] = df_slice['post_body'].progress_apply(analyze_post_for_metaphors)

# Count results
metaphors_found = sum(1 for result in df_slice['metaphor_analysis'] 
                     if isinstance(result, dict) and result.get('metaphor_present', False))
total_metaphor_instances = sum(len(result.get('metaphors', [])) 
                             for result in df_slice['metaphor_analysis'] if isinstance(result, dict))

print(f"\nResults Summary:")
print(f"- Posts analyzed: {len(df_slice)}")
print(f"- Posts with metaphors: {metaphors_found}")
print(f"- Total metaphor instances: {total_metaphor_instances}")

if len(df_slice) > 0:
    rate = metaphors_found / len(df_slice) * 100
    print(f"- Metaphor detection rate: {rate:.1f}%")

In [ ]:
# Cell 4: Save and inspect results
# Save results to JSON

output_data = {
    "processing_info": {
        "num_posts_analyzed": num_rows_to_process,
        "target_usas_tag": target_usas_tag,
        "model_used": model,
        "posts_with_metaphors": metaphors_found,
        "total_metaphor_instances": total_metaphor_instances
    },
    "results": df_slice[['post_body', 'metaphor_analysis']].to_dict('records')
}

output_filename = f"metaphor_analysis_direct_{num_rows_to_process}_posts.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Results saved to: {output_filename}")
# Display the first few results to verify
print("\nSample of results being saved:")
print(df_slice[['nid', 'post_body', 'metaphor_analysis']].head())

print("\n" + "="*80)
# Save to Excel for easy inspection
excel_filename = f"metaphor_analysis_{num_rows_to_process}_posts_2.xlsx"
df_slice.to_excel(excel_filename, index=False)
print(f"Data with analysis saved to: {excel_filename}")

In [ ]:
# Cell 5: Display sample findings
print("Sample metaphor findings:")
print("=" * 80)

sample_count = 0
for idx, row in df_slice.iterrows():
    result = row['metaphor_analysis']
    
    if result.get('metaphor_present', False) and sample_count < 3:
        post_id = row.get('id', idx)
        post_body = row['post_body']
        metaphors = result.get('metaphors', [])
        
        print(f"\nPost ID: {post_id}")
        print(f"Post excerpt: \"{post_body[:150]}...\"")
        print(f"Metaphors found: {len(metaphors)}")
        
        for i, metaphor in enumerate(metaphors):
            tenor = metaphor.get('tenor', {})
            vehicle = metaphor.get('vehicle', {})
            context = metaphor.get('context', '')
            
            print(f"  Metaphor {i+1}:")
            print(f"    Context: \"{context}\"")
            print(f"    Tenor: {tenor.get('concept', 'N/A')} (USAS: {tenor.get('usas_tag', 'N/A')})")
            print(f"    Vehicle: {vehicle.get('concept', 'N/A')} (USAS: {vehicle.get('usas_tag', 'N/A')})")
            print(f"    Tenor explicit: {tenor.get('explicit', 'N/A')}")
        
        sample_count += 1

if sample_count == 0:
    print("No metaphors found in the analyzed posts.")

In [ ]:
survey_data_path = "/work/speech/Metaphor data/Survey/dataset_2140253_20250308_15671279322069612044_long.xlsx"
survey_data_col = "answer_clean"
min_text_length_char = 15  # Minimum length of the input text to analyze

# Load the survey data from the specified Excel file
print(f"Loading survey data from: {survey_data_path}")
survey_df = pd.read_excel(survey_data_path)
print(f"Loaded {len(survey_df)} survey entries.")

# Ensure the target column exists and fill any missing values with an empty string
if survey_data_col not in survey_df.columns:
    raise ValueError(f"Column '{survey_data_col}' not found in the survey data.")
survey_df[survey_data_col] = survey_df[survey_data_col].fillna('').astype(str)

# Define a wrapper function to analyze survey answers, skipping short ones
def analyze_survey_answer(text):
    """
    Analyzes a single survey answer for metaphors, skipping answers that are too short.
    """
    # Skip analysis if the input text is shorter than the minimum required length
    if len(text.strip()) < min_text_length_char:
        return {'metaphor_present': False, 'skipped': True, 'reason': 'Input text too short'}
    
    # Reuse the main analysis function defined in Cell 3
    return analyze_post_for_metaphors(text)

# Apply the analysis function to the specified column in the survey DataFrame
print(f"\nAnalyzing column '{survey_data_col}' for metaphors...")
print(f"Skipping answers shorter than {min_text_length_char} characters.")

# Use .loc to create the new column and avoid SettingWithCopyWarning
survey_df.loc[:, 'metaphor_analysis'] = survey_df[survey_data_col].progress_apply(analyze_survey_answer)

# Display the first few results to verify
print("\nSample of survey analysis results:")
print(survey_df[[survey_data_col, 'metaphor_analysis']].head())

# Save the results to a new Excel file
survey_output_filename = "survey_metaphor_analysis_results.xlsx"
survey_df.to_excel(survey_output_filename, index=False)
print(f"\nSurvey analysis results saved to: {survey_output_filename}")


In [ ]:
# Process only first 50 rows
!python danish_metaphor_analysis.py --limit 5